# 🇧🇩 Bangladesh Economic Indicators — Full Dashboard v3


**Install:** `pip install dash plotly pandas scikit-learn kaleido` These Libray needed to run this Project.

In [1]:
# !pip install dash plotly pandas scikit-learn kaleido

In [2]:
import pandas as pd
import numpy as np
import dash
from dash import html, dcc
from dash.dependencies import Input, Output, State
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')
print('✅ All libraries loaded')

✅ All libraries loaded


In [3]:
# ── Load Datasets ─────────────────────────────────────────────────────────────
df = pd.read_csv('bangladesh_economy.csv')
nb = pd.read_csv('neighbors_data.csv')
print(f'Bangladesh data: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Neighbors data:  {nb.shape[0]} rows')
df.head(3)

Bangladesh data: 32 rows, 19 columns
Neighbors data:  48 rows


,Year,GDP_Billion_USD,GDP_Growth_Rate,GDP_Per_Capita_USD,Inflation_Rate,Unemployment_Rate,Remittance_Billion_USD,Export_Billion_USD,Import_Billion_USD,Population_Million,Poverty_Rate,RMG_Export_Billion_USD,Foreign_Reserve_Billion_USD,Interest_Rate,Tax_Revenue_GDP,Electricity_Coverage,Internet_Users_Million,CO2_Emissions_MT,Rice_Production_MT
0,1995,37.2,5.1,310.0,8.9,3.5,1.2,3.5,5.7,119.0,58.8,2.2,3.1,9.5,7.2,20.0,0.1,28.5,26.4
1,1996,40.5,4.6,331.0,6.7,3.5,1.4,3.9,6.1,120.1,56.7,2.5,3.3,9.2,7.4,22.0,0.2,29.8,26.7
2,1997,43.1,5.4,347.0,3.8,3.5,1.5,4.4,6.7,121.3,54.5,3.0,3.5,9.0,7.6,24.0,0.3,31.2,27.1


In [4]:
# ── ML Setup ──────────────────────────────────────────────────────────────────
PREDICT_UNTIL = 2034

INDICATORS = {
    'GDP_Billion_USD':          'GDP (Billion USD)',
    'GDP_Growth_Rate':          'GDP Growth Rate (%)',
    'Inflation_Rate':           'Inflation Rate (%)',
    'Unemployment_Rate':        'Unemployment Rate (%)',
    'Remittance_Billion_USD':   'Remittance (Billion USD)',
    'RMG_Export_Billion_USD':   'RMG Exports (Billion USD)',
    'Foreign_Reserve_Billion_USD': 'Foreign Reserves (Billion USD)',
}

def train_all_models(dataframe, column, until_year):
    """Train Linear, Polynomial, and Random Forest models. Return predictions + R2 scores."""
    X = dataframe[['Year']].values
    y = dataframe[column].values
    future_years = np.arange(dataframe['Year'].max() + 1, until_year + 1).reshape(-1, 1)

    models = {
        'Linear':     LinearRegression(),
        'Polynomial': make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    }

    results = {}
    for name, model in models.items():
        model.fit(X, y)
        train_preds = model.predict(X)
        r2 = r2_score(y, train_preds)
        future_preds = model.predict(future_years)
        # RF can't extrapolate well — nudge it slightly
        if name == 'Random Forest':
            trend = (y[-1] - y[-3]) / 2
            future_preds = future_preds + trend * np.arange(1, len(future_preds) + 1) * 0.3
        results[name] = {
            'future_years': future_years.flatten(),
            'predictions':  future_preds,
            'r2':           round(r2, 4),
        }
    return results

# Pre-train all models
ml_results = {
    col: train_all_models(df, col, PREDICT_UNTIL)
    for col in INDICATORS
}
print('✅ ML models trained:', list(INDICATORS.keys()))

✅ ML models trained: ['GDP_Billion_USD', 'GDP_Growth_Rate', 'Inflation_Rate', 'Unemployment_Rate', 'Remittance_Billion_USD', 'RMG_Export_Billion_USD', 'Foreign_Reserve_Billion_USD']


In [5]:
# ── Design System ─────────────────────────────────────────────────────────────
C = {
    'bg':           '#f8fafc',
    'card':         '#ffffff',
    'border':       '#e2e8f0',
    'green':        '#16a34a',
    'green_light':  '#dcfce7',
    'red':          '#dc2626',
    'red_light':    '#fee2e2',
    'blue':         '#2563eb',
    'blue_light':   '#dbeafe',
    'amber':        '#d97706',
    'amber_light':  '#fef3c7',
    'purple':       '#7c3aed',
    'purple_light': '#ede9fe',
    'teal':         '#0d9488',
    'text':         '#0f172a',
    'muted':        '#64748b',
    'grid':         '#f1f5f9',
}

CHART = dict(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(color=C['text'], family='Arial', size=12),
    xaxis=dict(gridcolor=C['grid'], showgrid=True, linecolor=C['border']),
    yaxis=dict(gridcolor=C['grid'], showgrid=True, linecolor=C['border']),
    legend=dict(bgcolor='rgba(0,0,0,0)', borderwidth=0),
    margin=dict(l=10, r=10, t=44, b=10),
    title_font=dict(size=14, color=C['text']),
)

MODEL_COLORS = {
    'Historical':   C['blue'],
    'Linear':       C['red'],
    'Polynomial':   C['green'],
    'Random Forest': C['purple'],
}

def th(fig):
    fig.update_layout(**CHART)
    return fig

def card(children, extra=None):
    s = {'background': C['card'], 'border': f'1px solid {C["border"]}',
         'borderRadius': '12px', 'padding': '20px'}
    if extra: s.update(extra)
    return html.Div(children, style=s)

def stat_card(label, value, color, bg):
    return html.Div([
        html.P(label, style={'color': C['muted'], 'fontSize': '12px',
                             'margin': '0 0 4px', 'fontWeight': '500'}),
        html.P(value, style={'color': color, 'fontSize': '20px',
                             'fontWeight': '700', 'margin': '0'}),
    ], style={'background': bg, 'borderRadius': '10px', 'padding': '14px 18px',
              'flex': '1', 'minWidth': '130px', 'border': f'1px solid {C["border"]}'})

def insight_banner(text, color, bg):
    return html.Div(
        html.P(f'💡 {text}', style={'margin': '0', 'fontSize': '13px',
                                    'color': color, 'fontWeight': '500'}),
        style={'background': bg, 'borderRadius': '8px', 'padding': '10px 16px',
               'marginBottom': '16px', 'border': f'1px solid {color}22'}
    )

def chart_card(fig, width='calc(50% - 8px)'):
    return html.Div(
        dcc.Graph(figure=fig, config={
            'displayModeBar': True,
            'toImageButtonOptions': {'format': 'png', 'filename': 'bd_chart',
                                     'height': 500, 'width': 900, 'scale': 2},
        }),
        style={'width': width, 'minWidth': '280px', 'background': C['card'],
               'border': f'1px solid {C["border"]}', 'borderRadius': '12px',
               'padding': '6px', 'boxSizing': 'border-box'}
    )

def grid(charts):
    return html.Div(charts, style={
        'display': 'flex', 'flexWrap': 'wrap',
        'gap': '16px', 'justifyContent': 'flex-start',
        'width': '100%', 'boxSizing': 'border-box'
    })


def yr(dataframe, year_range):
    ranges = {'all': (1995, 2026), '2000s': (2000, 2009),
              '2010s': (2010, 2019), 'recent': (2020, 2026)}
    s, e = ranges.get(year_range, (1995, 2026))
    return dataframe[(dataframe['Year'] >= s) & (dataframe['Year'] <= e)]

print('✅ Design system ready')

✅ Design system ready


In [6]:
# ── App Layout — Sidebar Navigation ──────────────────────────────────────────
app = dash.Dash(__name__, suppress_callback_exceptions=True)
app.title = 'Bangladesh Economy Dashboard'

NAV_TABS = [
    ('📈', 'GDP & Growth',       'gdp'),
    ('💸', 'Inflation & Jobs',   'inflation'),
    ('🌍', 'Trade & Remittance', 'trade'),
    ('⚡', 'Development',        'dev'),
    ('🌏', 'vs Neighbors',       'neighbors'),
    ('🔗', 'Correlations',       'correlation'),
    ('🤖', 'ML Predictions',     'ml'),
]

def nav_item(icon, label, value, active=False):
    return html.Div(
        id=f'nav-{value}',
        children=[
            html.Span(icon, style={'fontSize': '18px', 'width': '24px'}),
            html.Span(label, style={'fontSize': '13px', 'fontWeight': '600'}),
        ],
        style={
            'display': 'flex', 'alignItems': 'center', 'gap': '10px',
            'padding': '11px 16px', 'borderRadius': '8px', 'cursor': 'pointer',
            'background': C['green'] if active else 'transparent',
            'color': '#fff' if active else C['text'],
            'marginBottom': '4px',
            'transition': 'background 0.15s',
            'border': f'1px solid {C["green"]}' if active else '1px solid transparent',
        }
    )

app.layout = html.Div(
    style={'backgroundColor': C['bg'], 'minHeight': '100vh',
           'fontFamily': 'Arial, sans-serif', 'display': 'flex', 'flexDirection': 'column'},
    children=[

        # ── Top Header Bar ──
        html.Div(style={
            'background': f'linear-gradient(135deg, {C["green"]} 0%, #14532d 100%)',
            'padding': '16px 28px',
            'borderBottom': f'3px solid {C["red"]}',
            'display': 'flex', 'justifyContent': 'space-between',
            'alignItems': 'center', 'flexShrink': '0',
        }, children=[
            html.Div([
                html.H1('🇧🇩 Bangladesh Economic Indicators',
                        style={'color': '#fff', 'fontSize': '22px',
                               'fontWeight': '800', 'margin': '0 0 2px'}),
                html.P('Historical Data (1995–2026)  ·  ML Forecasts (2027–2034)  ·  South Asia Comparison',
                       style={'color': 'rgba(255,255,255,0.75)', 'fontSize': '12px', 'margin': '0'}),
            ]),
            html.Div(style={'display': 'flex', 'gap': '10px', 'alignItems': 'center'}, children=[
                html.Button('⬇ Export PNG', id='export-btn', n_clicks=0,
                            style={'background': 'rgba(255,255,255,0.15)', 'color': '#fff',
                                   'border': '1px solid rgba(255,255,255,0.4)',
                                   'borderRadius': '8px', 'padding': '7px 16px',
                                   'cursor': 'pointer', 'fontSize': '13px', 'fontWeight': '600'}),
                dcc.Download(id='download-png'),
            ]),
        ]),

        # ── Stat Cards Row ──
        html.Div(style={'padding': '14px 28px', 'borderBottom': f'1px solid {C["border"]}',
                        'background': C['card']}, children=[
            html.Div(id='stat-cards',
                     style={'display': 'flex', 'gap': '10px', 'flexWrap': 'wrap'}),
        ]),

        # ── Body: Sidebar + Content ──
        html.Div(style={'display': 'flex', 'flex': '1', 'overflow': 'hidden'}, children=[

            # ── Sidebar ──
            html.Div(style={
                'width': '220px', 'minWidth': '220px',
                'background': C['card'],
                'borderRight': f'1px solid {C["border"]}',
                'padding': '20px 12px',
                'display': 'flex', 'flexDirection': 'column',
                'gap': '2px',
            }, children=[

                html.P('NAVIGATION', style={
                    'color': C['muted'], 'fontSize': '10px', 'fontWeight': '700',
                    'letterSpacing': '0.1em', 'margin': '0 0 10px 8px',
                }),

                # Nav items
                *[html.Div(
                    children=[
                        html.Span(icon, style={'fontSize': '16px', 'width': '22px'}),
                        html.Span(label, style={'fontSize': '13px', 'fontWeight': '500'}),
                    ],
                    id={'type': 'nav-btn', 'index': value},
                    n_clicks=0,
                    style={
                        'display': 'flex', 'alignItems': 'center', 'gap': '10px',
                        'padding': '10px 14px', 'borderRadius': '8px', 'cursor': 'pointer',
                        'marginBottom': '2px', 'color': C['text'],
                        'border': '1px solid transparent',
                    }
                ) for icon, label, value in NAV_TABS],

                html.Hr(style={'border': 'none', 'borderTop': f'1px solid {C["border"]}',
                               'margin': '16px 0'}),

                # Year Range in sidebar
                html.P('YEAR RANGE', style={
                    'color': C['muted'], 'fontSize': '10px', 'fontWeight': '700',
                    'letterSpacing': '0.1em', 'margin': '0 0 8px 4px',
                }),
                dcc.Dropdown(
                    id='year-range',
                    options=[
                        {'label': 'All Years (1995–2026)', 'value': 'all'},
                        {'label': '2000s (2000–2009)',     'value': '2000s'},
                        {'label': '2010s (2010–2019)',     'value': '2010s'},
                        {'label': 'Recent (2020–2026)',    'value': 'recent'},
                    ],
                    value='all', clearable=False,
                    style={'fontSize': '12px'},
                ),

                html.Hr(style={'border': 'none', 'borderTop': f'1px solid {C["border"]}',
                               'margin': '16px 0'}),

                # What-if slider in sidebar
                html.Div([
                    html.P('WHAT-IF GROWTH %', style={
                        'color': C['muted'], 'fontSize': '10px', 'fontWeight': '700',
                        'letterSpacing': '0.1em', 'margin': '0 0 8px 4px',
                    }),
                    dcc.Slider(id='whatif-slider', min=1, max=10, step=0.5, value=5.0,
                               marks={i: f'{i}%' for i in [1, 3, 5, 7, 10]},
                               tooltip={'placement': 'bottom', 'always_visible': True}),
                ], id='whatif-controls', style={'display': 'none'}),

                # Hidden report-type store (driven by nav clicks)
                dcc.Store(id='report-type', data='gdp'),

                html.Div(style={'flex': '1'}),

                html.P('Data: World Bank & IMF',
                       style={'color': C['muted'], 'fontSize': '10px',
                              'textAlign': 'center', 'margin': '0'}),
            ]),

            # ── Main Content ──
            html.Div(style={'flex': '1', 'overflowY': 'auto', 'padding': '24px 28px'}, children=[

                # Section title
                html.Div(id='section-title', style={'marginBottom': '16px'}, children=[
                    html.H2('📈 GDP & Growth',
                            style={'color': C['text'], 'fontSize': '20px',
                                   'fontWeight': '700', 'margin': '0'}),
                ]),

                # Charts output
                html.Div(id='charts-container'),
            ]),
        ]),
    ],
)

In [7]:
# ── Callback: Handle sidebar nav clicks → update active tab ──────────────────
from dash.dependencies import ALL

@app.callback(
    Output('report-type', 'data'),
    Input({'type': 'nav-btn', 'index': ALL}, 'n_clicks'),
    prevent_initial_call=True,
)
def update_active_tab(n_clicks_list):
    ctx = dash.callback_context
    if not ctx.triggered:
        return 'gdp'
    triggered_id = ctx.triggered[0]['prop_id']
    import json
    id_dict = json.loads(triggered_id.split('.')[0])
    return id_dict['index']


# ── Callback: Highlight active nav item ──────────────────────────────────────
@app.callback(
    [Output({'type': 'nav-btn', 'index': val}, 'style') for _, _, val in [
        ('📈', 'GDP & Growth',       'gdp'),
        ('💸', 'Inflation & Jobs',   'inflation'),
        ('🌍', 'Trade & Remittance', 'trade'),
        ('⚡', 'Development',        'dev'),
        ('🌏', 'vs Neighbors',       'neighbors'),
        ('🔗', 'Correlations',       'correlation'),
        ('🤖', 'ML Predictions',     'ml'),
    ]],
    Input('report-type', 'data'),
)
def highlight_nav(active_tab):
    styles = []
    for _, _, val in [
        ('📈', 'GDP & Growth',       'gdp'),
        ('💸', 'Inflation & Jobs',   'inflation'),
        ('🌍', 'Trade & Remittance', 'trade'),
        ('⚡', 'Development',        'dev'),
        ('🌏', 'vs Neighbors',       'neighbors'),
        ('🔗', 'Correlations',       'correlation'),
        ('🤖', 'ML Predictions',     'ml'),
    ]:
        if val == active_tab:
            styles.append({
                'display': 'flex', 'alignItems': 'center', 'gap': '10px',
                'padding': '10px 14px', 'borderRadius': '8px', 'cursor': 'pointer',
                'marginBottom': '2px', 'color': '#fff',
                'background': '#16a34a',
                'border': '1px solid #16a34a',
                'fontWeight': '600',
            })
        else:
            styles.append({
                'display': 'flex', 'alignItems': 'center', 'gap': '10px',
                'padding': '10px 14px', 'borderRadius': '8px', 'cursor': 'pointer',
                'marginBottom': '2px', 'color': '#0f172a',
                'background': 'transparent',
                'border': '1px solid transparent',
            })
    return styles


# ── Callback: Update section title ───────────────────────────────────────────
@app.callback(
    Output('section-title', 'children'),
    Input('report-type', 'data'),
)
def update_section_title(tab):
    labels = {
        'gdp':         '📈 GDP & Growth',
        'inflation':   '💸 Inflation & Jobs',
        'trade':       '🌍 Trade & Remittance',
        'dev':         '⚡ Development Indicators',
        'neighbors':   '🌏 Bangladesh vs Neighbors',
        'correlation': '🔗 Correlations',
        'ml':          '🤖 ML Predictions (2027–2034)',
    }
    return html.H2(labels.get(tab, ''),
                   style={'color': '#0f172a', 'fontSize': '20px',
                          'fontWeight': '700', 'margin': '0'})


# ── Callback: Show what-if slider only on ML tab ──────────────────────────────
@app.callback(
    Output('whatif-controls', 'style'),
    Input('report-type', 'data'),
)
def toggle_whatif(tab):
    if tab == 'ml':
        return {'display': 'block'}
    return {'display': 'none'}


# ── Callback: Dynamic stat cards ─────────────────────────────────────────────
@app.callback(
    Output('stat-cards', 'children'),
    Input('year-range', 'value'),
)
def update_stat_cards(year_range):
    fd = yr(df, year_range)
    latest = fd.iloc[-1]
    first  = fd.iloc[0]
    gdp_growth_pct = ((latest['GDP_Billion_USD'] - first['GDP_Billion_USD'])
                      / first['GDP_Billion_USD'] * 100)
    return [
        stat_card(f'GDP {int(latest["Year"])}',
                  f'${latest["GDP_Billion_USD"]:.0f}B',       C['green'],  C['green_light']),
        stat_card(f'Growth {int(latest["Year"])}',
                  f'{latest["GDP_Growth_Rate"]}%',             C['blue'],   C['blue_light']),
        stat_card(f'Inflation {int(latest["Year"])}',
                  f'{latest["Inflation_Rate"]}%',              C['red'],    C['red_light']),
        stat_card(f'Unemployment',
                  f'{latest["Unemployment_Rate"]}%',           C['amber'],  C['amber_light']),
        stat_card(f'Remittance {int(latest["Year"])}',
                  f'${latest["Remittance_Billion_USD"]}B',     C['teal'],   '#ccfbf1'),
        stat_card(f'RMG Exports {int(latest["Year"])}',
                  f'${latest["RMG_Export_Billion_USD"]}B',     C['purple'], C['purple_light']),
        stat_card(f'Forex Reserves',
                  f'${latest["Foreign_Reserve_Billion_USD"]}B', C['blue'],  C['blue_light']),
        stat_card(f'Range GDP Growth',
                  f'{gdp_growth_pct:.1f}%',                    C['green'],  C['green_light']),
    ]

In [8]:
# ── Main Callback: Render charts ──────────────────────────────────────────────
@app.callback(
    Output('charts-container', 'children'),
    [Input('report-type', 'data'),
     Input('year-range',  'value'),
     Input('whatif-slider', 'value')],
    prevent_initial_call=False,
)
def update_charts(tab, year_range, whatif_rate):
    whatif_rate = whatif_rate if whatif_rate is not None else 5.0
    fd = yr(df, year_range)
    y_start = int(fd['Year'].iloc[0])
    y_end   = int(fd['Year'].iloc[-1])

    # ════════════════════════════════════════════════════════
    # TAB 1 — GDP & Growth
    # ════════════════════════════════════════════════════════
    if tab == 'gdp':
        f1 = th(px.area(fd, x='Year', y='GDP_Billion_USD',
                        title='GDP Over Time (Billion USD)',
                        color_discrete_sequence=[C['green']]))
        f2 = th(px.bar(fd, x='Year', y='GDP_Growth_Rate',
                       title='Annual GDP Growth Rate (%)',
                       color='GDP_Growth_Rate',
                       color_continuous_scale=[[0,'#ef4444'],[0.4,'#fbbf24'],[1,'#22c55e']]))
        f3 = th(px.line(fd, x='Year', y='GDP_Per_Capita_USD',
                        title='GDP Per Capita (USD)', markers=True,
                        color_discrete_sequence=[C['blue']]))
        f4 = th(px.scatter(fd, x='GDP_Growth_Rate', y='GDP_Per_Capita_USD',
                           size='GDP_Billion_USD', color='Year',
                           title='Growth Rate vs Per Capita (bubble = GDP size)',
                           color_continuous_scale='Blues'))
        ddf = fd.copy()
        ddf['Decade'] = (ddf['Year'] // 10 * 10).astype(str) + 's'
        decade_avg = ddf.groupby('Decade')[['GDP_Growth_Rate','Inflation_Rate','Unemployment_Rate']].mean().reset_index()
        f5 = th(px.bar(decade_avg, x='Decade',
                       y=['GDP_Growth_Rate','Inflation_Rate','Unemployment_Rate'],
                       title='Decade Averages: Growth vs Inflation vs Unemployment',
                       barmode='group',
                       color_discrete_sequence=[C['green'], C['red'], C['amber']]))
        f6 = go.Figure()
        f6.add_trace(go.Scatter(x=fd['Year'], y=fd['GDP_Billion_USD'],
                                mode='lines', fill='tozeroy',
                                fillcolor='rgba(22,163,74,0.1)',
                                line=dict(color=C['green'], width=2.5)))
        for yr_val, label in [(2006,'$61B: Garments boom'),(2015,'$195B: Digital BD'),(2021,'$355B: Post-COVID')]:
            row = fd[fd['Year']==yr_val]
            if not row.empty:
                f6.add_annotation(x=yr_val, y=row['GDP_Billion_USD'].values[0], text=label,
                                  showarrow=True, arrowhead=2, arrowcolor=C['muted'],
                                  font=dict(size=10, color=C['muted']),
                                  bgcolor='white', bordercolor=C['border'])
        f6.update_layout(title='GDP Journey with Key Milestones', **CHART)
        ins = insight_banner(
            f"Bangladesh's GDP grew {fd['GDP_Billion_USD'].iloc[-1]/fd['GDP_Billion_USD'].iloc[0]:.1f}x "
            f"from {y_start} to {y_end} — average growth of {fd['GDP_Growth_Rate'].mean():.1f}% per year.",
            C['green'], C['green_light'])
        return html.Div([ins, grid([chart_card(f1), chart_card(f2),
                                    chart_card(f3), chart_card(f4),
                                    chart_card(f5), chart_card(f6)])])

    # ════════════════════════════════════════════════════════
    # TAB 2 — Inflation & Jobs
    # ════════════════════════════════════════════════════════
    elif tab == 'inflation':
        f1 = th(px.line(fd, x='Year', y='Inflation_Rate',
                        title='Inflation Rate (%)', markers=True,
                        color_discrete_sequence=[C['red']]))
        f1.add_hline(y=5, line_dash='dash', line_color=C['muted'],
                     annotation_text='5% target', annotation_font_color=C['muted'])
        f2 = th(px.bar(fd, x='Year', y='Unemployment_Rate',
                       title='Unemployment Rate (%)',
                       color='Unemployment_Rate',
                       color_continuous_scale=[[0, C['green']], [1, C['red']]]))
        f3 = th(px.scatter(fd, x='Inflation_Rate', y='Unemployment_Rate',
                           text='Year', color='Year',
                           title='Phillips Curve: Inflation vs Unemployment',
                           color_continuous_scale='RdYlGn_r'))
        f4 = th(px.line(fd, x='Year', y=['Inflation_Rate','Unemployment_Rate'],
                        title='Inflation vs Unemployment Trend', markers=True,
                        color_discrete_sequence=[C['red'], C['amber']]))
        f5 = th(px.area(fd, x='Year', y='Poverty_Rate',
                        title='Poverty Rate Over Time (%)',
                        color_discrete_sequence=[C['amber']]))
        f6 = th(px.scatter(fd, x='Inflation_Rate', y='Poverty_Rate',
                           color='Year', size='Unemployment_Rate',
                           title='Inflation vs Poverty Rate',
                           color_continuous_scale='YlOrRd'))
        poverty_drop = fd['Poverty_Rate'].iloc[0] - fd['Poverty_Rate'].iloc[-1]
        ins = insight_banner(
            f"Poverty dropped from {fd['Poverty_Rate'].iloc[0]}% in {y_start} to "
            f"{fd['Poverty_Rate'].iloc[-1]}% in {y_end} — a {poverty_drop:.1f}pp reduction "
            f"despite average inflation of {fd['Inflation_Rate'].mean():.1f}%.",
            C['amber'], C['amber_light'])
        return html.Div([ins, grid([chart_card(f1), chart_card(f2),
                                    chart_card(f3), chart_card(f4),
                                    chart_card(f5), chart_card(f6)])])

    # ════════════════════════════════════════════════════════
    # TAB 3 — Trade & Remittance
    # ════════════════════════════════════════════════════════
    elif tab == 'trade':
        f1 = th(px.line(fd, x='Year', y=['Export_Billion_USD','Import_Billion_USD'],
                        title='Exports vs Imports (Billion USD)', markers=True,
                        color_discrete_sequence=[C['green'], C['red']]))
        fd2 = fd.copy()
        fd2['Trade_Balance'] = fd2['Export_Billion_USD'] - fd2['Import_Billion_USD']
        f2 = th(px.bar(fd2, x='Year', y='Trade_Balance',
                       title='Trade Balance (Exports − Imports)',
                       color='Trade_Balance',
                       color_continuous_scale=[[0,C['red']],[0.5,C['amber']],[1,C['green']]]))
        f3 = th(px.area(fd, x='Year', y='Remittance_Billion_USD',
                        title='Remittance Inflow (Billion USD)',
                        color_discrete_sequence=[C['purple']]))
        f4 = th(px.area(fd, x='Year', y='RMG_Export_Billion_USD',
                        title='Ready-Made Garments (RMG) Export (Billion USD)',
                        color_discrete_sequence=[C['teal']]))
        f5 = th(px.bar(fd, x='Year',
                       y=['Export_Billion_USD','Import_Billion_USD','Remittance_Billion_USD'],
                       title='Export vs Import vs Remittance Comparison',
                       barmode='group',
                       color_discrete_sequence=[C['green'], C['red'], C['purple']]))
        fd3 = fd.copy()
        fd3['RMG_Share'] = (fd3['RMG_Export_Billion_USD'] / fd3['Export_Billion_USD'] * 100).round(1)
        f6 = th(px.line(fd3, x='Year', y='RMG_Share',
                        title='RMG Share of Total Exports (%)', markers=True,
                        color_discrete_sequence=[C['teal']]))
        f6.add_hline(y=80, line_dash='dash', line_color=C['muted'],
                     annotation_text='80% mark', annotation_font_color=C['muted'])
        remit_growth = fd['Remittance_Billion_USD'].iloc[-1] / fd['Remittance_Billion_USD'].iloc[0]
        ins = insight_banner(
            f"RMG exports reached ${fd['RMG_Export_Billion_USD'].iloc[-1]}B in {y_end}, "
            f"making up over 80% of total exports. Remittances grew "
            f"{remit_growth:.0f}x from {y_start} to {y_end}.",
            C['teal'], '#ccfbf1')
        return html.Div([ins, grid([chart_card(f1), chart_card(f2),
                                    chart_card(f3), chart_card(f4),
                                    chart_card(f5), chart_card(f6)])])

    # ════════════════════════════════════════════════════════
    # TAB 4 — Development Indicators
    # ════════════════════════════════════════════════════════
    elif tab == 'dev':
        f1 = th(px.line(fd, x='Year', y='Electricity_Coverage',
                        title='Electricity Coverage (%)', markers=True,
                        color_discrete_sequence=[C['amber']]))
        f1.add_hline(y=100, line_dash='dash', line_color=C['green'],
                     annotation_text='100% achieved 2022')
        f2 = th(px.area(fd, x='Year', y='Internet_Users_Million',
                        title='Internet Users (Millions)',
                        color_discrete_sequence=[C['blue']]))
        f3 = th(px.line(fd, x='Year', y='Foreign_Reserve_Billion_USD',
                        title='Foreign Exchange Reserves (Billion USD)', markers=True,
                        color_discrete_sequence=[C['green']]))
        f4 = th(px.bar(fd, x='Year', y='CO2_Emissions_MT',
                       title='CO₂ Emissions (Million Tonnes)',
                       color='CO2_Emissions_MT',
                       color_continuous_scale=[[0, C['green_light']], [1, C['red']]]))
        f5 = th(px.line(fd, x='Year', y='Rice_Production_MT',
                        title='Rice Production (Million Tonnes)', markers=True,
                        color_discrete_sequence=[C['amber']]))
        f6 = th(px.line(fd, x='Year', y=['Electricity_Coverage','Internet_Users_Million'],
                        title='Electricity Coverage vs Internet Users', markers=True,
                        color_discrete_sequence=[C['amber'], C['blue']]))
        ins = insight_banner(
            f"Bangladesh achieved 100% electricity coverage in 2022 and grew internet "
            f"users from {fd['Internet_Users_Million'].iloc[0]}M in {y_start} to "
            f"{fd['Internet_Users_Million'].iloc[-1]}M in {y_end}.",
            C['blue'], C['blue_light'])
        return html.Div([ins, grid([chart_card(f1), chart_card(f2),
                                    chart_card(f3), chart_card(f4),
                                    chart_card(f5), chart_card(f6)])])

    # ════════════════════════════════════════════════════════
    # TAB 5 — vs Neighbors
    # ════════════════════════════════════════════════════════
    elif tab == 'neighbors':
        COUNTRY_COLORS = {
            'Bangladesh': C['green'], 'India': C['amber'],
            'Pakistan': C['blue'], 'Sri Lanka': C['purple'],
            'Nepal': C['teal'], 'Myanmar': C['red'],
        }
        # Filter neighbors by selected year range
        nb_fd = nb[(nb['Year'] >= y_start) & (nb['Year'] <= y_end)]
        if nb_fd.empty:
            nb_fd = nb

        f1 = th(px.line(nb_fd, x='Year', y='GDP_Per_Capita_USD', color='Country',
                        title=f'GDP Per Capita Comparison — South Asia ({y_start}–{y_end})',
                        markers=True, color_discrete_map=COUNTRY_COLORS))
        f2 = th(px.bar(nb_fd, x='Country', y='GDP_Per_Capita_USD',
                       animation_frame='Year',
                       title=f'GDP Per Capita Race — animated ({y_start}–{y_end})',
                       color='Country', color_discrete_map=COUNTRY_COLORS))
        f3 = th(px.line(nb_fd, x='Year', y='GDP_Growth_Rate', color='Country',
                        title=f'GDP Growth Rate Comparison % ({y_start}–{y_end})',
                        markers=True, color_discrete_map=COUNTRY_COLORS))
        f4 = th(px.line(nb_fd, x='Year', y='Inflation_Rate', color='Country',
                        title=f'Inflation Rate Comparison % ({y_start}–{y_end})',
                        markers=True, color_discrete_map=COUNTRY_COLORS))

        # Radar — use latest year in filtered range
        latest_year = int(nb_fd['Year'].max())
        latest_nb   = nb_fd[nb_fd['Year'] == latest_year].copy()
        f5 = go.Figure()
        cats = ['GDP_Per_Capita_USD', 'GDP_Growth_Rate', 'Inflation_Rate', 'Unemployment_Rate']
        cat_labels = ['GDP/Capita', 'GDP Growth', 'Inflation', 'Unemployment']
        for _, row in latest_nb.iterrows():
            vals  = [row[c] for c in cats]
            denom = [(nb[c].max() - nb[c].min()) for c in cats]
            vals_norm = [(v - nb[c].min()) / d * 100 if d > 0 else 0
                         for v, c, d in zip(vals, cats, denom)]
            f5.add_trace(go.Scatterpolar(
                r=vals_norm + [vals_norm[0]],
                theta=cat_labels + [cat_labels[0]],
                name=row['Country'],
                line=dict(color=COUNTRY_COLORS.get(row['Country'], '#888'))
            ))
        f5.update_layout(title=f'{latest_year} Country Comparison (Normalized)',
                         polar=dict(bgcolor='rgba(0,0,0,0)'), **CHART)

        # Rank chart — filter by range too
        rank_df = nb_fd.copy()
        rank_df['Rank'] = rank_df.groupby('Year')['GDP_Per_Capita_USD'].rank(ascending=False)
        bd_rank = rank_df[rank_df['Country'] == 'Bangladesh'].copy()
        f6 = th(px.line(bd_rank, x='Year', y='Rank',
                        title=f'Bangladesh Rank in South Asia ({y_start}–{y_end}, lower = better)',
                        markers=True, color_discrete_sequence=[C['green']]))
        f6.update_yaxes(autorange='reversed', tickvals=[1,2,3,4,5,6])

        # Dynamic insight
        bd_latest   = nb_fd[(nb_fd['Country']=='Bangladesh') & (nb_fd['Year']==latest_year)]
        bd_gdp      = bd_latest['GDP_Per_Capita_USD'].values[0]
        bd_rank_val = int(rank_df[(rank_df['Country']=='Bangladesh') &
                                   (rank_df['Year']==latest_year)]['Rank'].values[0])
        ins = insight_banner(
            f"In {latest_year}, Bangladesh GDP per capita was ${bd_gdp:,.0f} "
            f"— ranked #{bd_rank_val} among South Asian neighbors. "
            f"Showing {y_start}–{y_end} data.",
            C['green'], C['green_light'])
        return html.Div([ins, grid([chart_card(f1), chart_card(f2),
                                    chart_card(f3), chart_card(f4),
                                    chart_card(f5), chart_card(f6)])])

    # ════════════════════════════════════════════════════════
    # TAB 6 — Correlations
    # ════════════════════════════════════════════════════════
    elif tab == 'correlation':
        num_cols = ['GDP_Billion_USD','GDP_Growth_Rate','Inflation_Rate',
                    'Unemployment_Rate','Remittance_Billion_USD',
                    'Export_Billion_USD','Import_Billion_USD','Poverty_Rate',
                    'Electricity_Coverage','Internet_Users_Million','CO2_Emissions_MT']
        corr = fd[num_cols].corr().round(2)
        short = ['GDP','GDP Gr.','Inflation','Unemploy.','Remittance',
                 'Export','Import','Poverty','Electricity','Internet','CO₂']
        f1 = go.Figure(go.Heatmap(
            z=corr.values, x=short, y=short,
            colorscale=[[0,C['red']],[0.5,'#f8fafc'],[1,C['green']]],
            zmid=0, text=corr.values.round(2),
            texttemplate='%{text}', textfont=dict(size=10),
            hoverongaps=False,
        ))
        f1.update_layout(title=f'Correlation Heatmap ({y_start}–{y_end})', **CHART)
        f2 = th(px.scatter(fd, x='GDP_Per_Capita_USD', y='Poverty_Rate',
                           color='Year', size='Population_Million',
                           title='GDP Per Capita vs Poverty Rate',
                           trendline='ols',
                           color_continuous_scale='Blues'))
        f3 = th(px.scatter(fd, x='GDP_Billion_USD', y='CO2_Emissions_MT',
                           color='Year', size='Population_Million',
                           title='GDP Growth vs CO₂ Emissions',
                           trendline='ols',
                           color_continuous_scale='Reds'))
        f4 = th(px.scatter(fd, x='Remittance_Billion_USD', y='GDP_Billion_USD',
                           color='Year', size='Import_Billion_USD',
                           title='Remittance vs GDP (bubble = imports)',
                           trendline='ols',
                           color_continuous_scale='Greens'))
        corr_val = fd['GDP_Per_Capita_USD'].corr(fd['Poverty_Rate'])
        ins = insight_banner(
            f"GDP per capita has a strong negative correlation (r = {corr_val:.2f}) with poverty rate "
            f"from {y_start}–{y_end}, confirming economic growth directly lifts people out of poverty.",
            C['purple'], C['purple_light'])
        return html.Div([ins, grid([chart_card(f1, width='98%'),
                                    chart_card(f2), chart_card(f3), chart_card(f4)])])

    # ════════════════════════════════════════════════════════
    # TAB 7 — ML Predictions
    # ════════════════════════════════════════════════════════
    elif tab == 'ml':
        last_gdp  = df['GDP_Billion_USD'].iloc[-1]
        last_year = int(df['Year'].iloc[-1])
        wi_years  = list(range(last_year + 1, last_year + 11))
        wi_gdp    = [last_gdp * ((1 + whatif_rate/100)**i) for i in range(1, 11)]
        wi_fig = go.Figure()
        wi_fig.add_trace(go.Scatter(
            x=df['Year'], y=df['GDP_Billion_USD'],
            mode='lines+markers', name='Historical',
            line=dict(color=C['blue'], width=2.5)))
        wi_fig.add_trace(go.Scatter(
            x=wi_years, y=wi_gdp,
            mode='lines+markers', name=f'What-if @ {whatif_rate}% growth',
            line=dict(color=C['green'], width=2.5, dash='dash'),
            marker=dict(symbol='diamond', size=7)))
        wi_fig.add_vline(x=last_year + 0.5, line_dash='dot', line_color=C['muted'])
        wi_fig.update_layout(
            title=f'What-if Simulator: GDP if growth stays at {whatif_rate}% ({last_year+1}–{last_year+10})',
            **CHART)
        charts = [chart_card(th(wi_fig), width='98%')]
        r2_cards = []
        for col, label in list(INDICATORS.items())[:3]:
            scores = ml_results[col]
            r2_cards.append(html.Div([
                html.P(label, style={'fontWeight': '700', 'color': C['text'],
                                     'fontSize': '13px', 'margin': '0 0 8px'}),
                html.Div(style={'display': 'flex', 'gap': '8px'}, children=[
                    html.Span(f'Linear R²: {scores["Linear"]["r2"]}',
                              style={'background': C['red_light'], 'color': C['red'],
                                     'borderRadius': '6px', 'padding': '3px 8px',
                                     'fontSize': '12px', 'fontWeight': '600'}),
                    html.Span(f'Poly R²: {scores["Polynomial"]["r2"]}',
                              style={'background': C['green_light'], 'color': C['green'],
                                     'borderRadius': '6px', 'padding': '3px 8px',
                                     'fontSize': '12px', 'fontWeight': '600'}),
                    html.Span(f'RF R²: {scores["Random Forest"]["r2"]}',
                              style={'background': C['purple_light'], 'color': C['purple'],
                                     'borderRadius': '6px', 'padding': '3px 8px',
                                     'fontSize': '12px', 'fontWeight': '600'}),
                ]),
            ], style={'background': C['card'], 'border': f'1px solid {C["border"]}',
                      'borderRadius': '10px', 'padding': '14px', 'flex': '1', 'minWidth': '280px'}))
        r2_row = html.Div([
            html.P('Model Accuracy (R² Score — closer to 1.0 is better)',
                   style={'fontWeight': '700', 'color': C['text'],
                          'fontSize': '14px', 'margin': '0 0 12px'}),
            html.Div(r2_cards, style={'display': 'flex', 'gap': '12px', 'flexWrap': 'wrap'}),
        ], style={'background': C['bg'], 'border': f'1px solid {C["border"]}',
                  'borderRadius': '12px', 'padding': '16px', 'marginBottom': '16px'})
        for col, label in INDICATORS.items():
            fig = go.Figure()
            fig.add_trace(go.Scatter(
                x=df['Year'], y=df[col],
                mode='lines+markers', name='Historical',
                line=dict(color=MODEL_COLORS['Historical'], width=2.5)))
            for model_name in ['Linear', 'Polynomial', 'Random Forest']:
                res = ml_results[col][model_name]
                fig.add_trace(go.Scatter(
                    x=res['future_years'], y=res['predictions'],
                    mode='lines+markers', name=f'{model_name} (R²={res["r2"]})',
                    line=dict(color=MODEL_COLORS[model_name], width=2, dash='dash'),
                    marker=dict(size=5, symbol='diamond')))
            fig.add_vline(x=last_year + 0.5, line_dash='dot', line_color=C['muted'],
                          annotation_text='Forecast →',
                          annotation_font_color=C['muted'], annotation_font_size=11)
            fig.update_layout(title=label, **CHART)
            charts.append(chart_card(th(fig)))
        ins = insight_banner(
            f"What-if @ {whatif_rate}% growth: Bangladesh GDP could reach "
            f"${wi_gdp[-1]:.0f}B by {wi_years[-1]}. "
            f"Compare Linear, Polynomial and Random Forest forecasts above.",
            C['purple'], C['purple_light'])
        return html.Div([ins, r2_row, grid(charts)])

    return html.Div()

In [9]:
# ── Export Callback ───────────────────────────────────────────────────────────
@app.callback(
    Output('download-png', 'data'),
    Input('export-btn', 'n_clicks'),
    prevent_initial_call=True,
)
def export_png(n_clicks):
    import plotly.io as pio

    fig = make_subplots(
        rows=3, cols=3,
        subplot_titles=[
            'GDP (Billion USD)', 'GDP Growth Rate (%)', 'Inflation Rate (%)',
            'Unemployment Rate (%)', 'Remittance (Billion USD)', 'RMG Exports (Billion USD)',
            'Electricity Coverage (%)', 'Internet Users (M)', 'Poverty Rate (%)',
        ],
    )
    plots = [
        ('GDP_Billion_USD',         C['green'],  1,1),
        ('GDP_Growth_Rate',         C['blue'],   1,2),
        ('Inflation_Rate',          C['red'],    1,3),
        ('Unemployment_Rate',       C['amber'],  2,1),
        ('Remittance_Billion_USD',  C['purple'], 2,2),
        ('RMG_Export_Billion_USD',  C['teal'],   2,3),
        ('Electricity_Coverage',    C['amber'],  3,1),
        ('Internet_Users_Million',  C['blue'],   3,2),
        ('Poverty_Rate',            C['red'],    3,3),
    ]
    for col, color, row, c in plots:
        fig.add_trace(go.Scatter(
            x=df['Year'], y=df[col], mode='lines+markers',
            line=dict(color=color, width=2), marker=dict(size=4), showlegend=False),
            row=row, col=c)

    fig.update_layout(
        height=1000, width=1600,
        title_text='Bangladesh Economic Indicators — Full Summary (1995–2024)',
        title_font=dict(size=20, color=C['text']),
        paper_bgcolor='#f8fafc', plot_bgcolor='#ffffff',
        font=dict(color=C['text'], family='Arial'),
    )
    fig.update_xaxes(gridcolor=C['grid'])
    fig.update_yaxes(gridcolor=C['grid'])

    img_bytes = pio.to_image(fig, format='png', scale=2)
    return dcc.send_bytes(img_bytes, 'bangladesh_full_summary.png')

In [10]:
# ── Run ───────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    app.run(debug=True, port=8052)